In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# =========================
# CONFIG
# =========================
DATA_DIR = r"D:\CODE\MiniProject\ml-service\data\leaf_images\small_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# =========================
# LOAD DATA
# =========================
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)

# =========================
# PERFORMANCE
# =========================
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# =========================
# MODEL
# =========================
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # freeze base

model = models.Sequential([
    layers.Rescaling(1./255),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# =========================
# TRAIN
# =========================
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =========================
# SAVE MODEL
# =========================
model.save(r"D:\CODE\MiniProject\ml-service\models\leaf_model.h5")
print("✅ Model saved!")


Found 2000 files belonging to 5 classes.
Using 1600 files for training.
Found 2000 files belonging to 5 classes.
Using 400 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 52s 828ms/step - accuracy: 0.6381 - loss: 0.9179 - val_accuracy: 0.8550 - val_loss: 0.4243
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 36s 725ms/step - accuracy: 0.8313 - loss: 0.4549 - val_accuracy: 0.8825 - val_loss: 0.3167
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 38s 766ms/step - accuracy: 0.8694 - loss: 0.3513 - val_accuracy: 0.8925 - val_loss: 0.2994
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 40s 800ms/step - accuracy: 0.8956 - loss: 0.3079 - val_accuracy: 0.8900 - val_loss: 0.2976
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 38s 763ms/step - accuracy: 0.9044 - loss: 0.2628 - val_accuracy: 0.9200 - val_loss: 0.2353
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 39s 784ms/step - accuracy: 0.9206 - loss: 0.2190 - val_accuracy: 0.9100 - val_loss: 0.2501
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 37s 731ms/step - accuracy: 0.9337 - loss: 0.1859 - val_accuracy: 0.9200 - val_loss: 0.2141
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 37s 733ms/step - accuracy: 0.9538 - loss: 0.1588 - val_accu

✅ Model saved!


In [2]:
import json
import os

SAVE_DIR = r"D:\CODE\MiniProject\ml-service\models"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save class names
with open(os.path.join(SAVE_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f)

print("✅ Class names saved:", class_names)


✅ Class names saved: ['Tomato_Bacterial_Spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_healthy']


In [3]:
import tensorflow as tf

MODEL_PATH = r"D:\CODE\MiniProject\ml-service\models\leaf_model.h5"

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()
base_model = model.layers[1]  # MobileNetV2

base_model.trainable = True

# Freeze early layers (keep learned low-level features)
for layer in base_model.layers[:100]:
    layer.trainable = False

print("Trainable layers:")
for layer in base_model.layers[-10:]:
    print(layer.name, layer.trainable)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)
model.save(r"D:\CODE\MiniProject\ml-service\models\leaf_model_finetuned.h5")
print("✅ Fine-tuned model saved!")



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,599 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 2 (12.00 B)

Trainable layers:
block_16_expand_BN True
block_16_expand_relu True
block_16_depthwise True
block_16_depthwise_BN True
block_16_depthwise_relu True
block_16_project True
block_16_project_BN True
Conv_1 True
Conv_1_bn True
out_relu True
Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.5312 - loss: 2.0211 - val_accuracy: 0.9275 - val_loss: 0.1910
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.7669 - loss: 0.7160 - val_accuracy: 0.9200 - val_loss: 0.2049
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 62s 1s/step - accuracy: 0.8750 - loss: 0.3393 - val_accuracy: 0.9275 - val_loss: 0.1955
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 61s 1s/step - accuracy: 0.9125 - loss: 0.2535 - val_accuracy: 0.9275 - val_loss: 0.1820
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.9275 - loss: 0.1938 - val_accuracy: 0.9400 - val_loss: 0.1752


✅ Fine-tuned model saved!
